# Stage1 × ADS 历史实时版本可用性

冻结来源：费城联储 ADS 全部版本原件，NAS 已验证恢复。仅沿原 Stage1 的 Qlib 日历检查每个决策日严格早于该日的最新版本与版本日以内的最新观测。

执行预算、停止条件见 `plan.json`。本卡尚未执行；0 ADS 数值、0 市场结果、0 拟合、0 账户。版本日不代表盘中发布时钟。


In [1]:
import os,json,hashlib,zipfile,tempfile,shutil,math,xml.etree.ElementTree as ET
from datetime import datetime
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path.cwd();card=root/'research/stage1_ads_availability_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024*1024),b''):h.update(chunk)
 return h.hexdigest()
source=root/plan['source']['raw_path']
assert plan['frozen'] and sha(source)==plan['source']['raw_sha256']
src=json.loads((root/'research/philadelphia_ads_vintage_source_20261023/source_preflight_result.json').read_text())
assert src['nas_archive_sha256']==plan['source']['NAS_archive_sha256'] and src['scoped_raw_ZIP_restore_verified']
assert json.loads((root/plan['source']['schema_audit']).read_text())['vintage_column_count']==1576
cache=root/'data/snapshots/philadelphia_ads_vintage_20261023/ads_diagonal_asof_vintage.json'
assert not cache.exists(), 'one source pass already materialized; inspect state before rerun'
ns='{http://schemas.openxmlformats.org/spreadsheetml/2006/main}'
with zipfile.ZipFile(source) as outer:
 assert outer.namelist()==[plan['source']['sheet'].replace('Sheet1','ADS_All_Vintages-zip.xlsx')]
 with outer.open(outer.namelist()[0]) as inp,tempfile.NamedTemporaryFile(suffix='.xlsx',delete=False) as dst:
  shutil.copyfileobj(inp,dst); tmp=Path(dst.name)
try:
 with zipfile.ZipFile(tmp) as book:
  strs=[''.join(t.text or '' for t in si.iter(ns+'t')) for si in ET.fromstring(book.read('xl/sharedStrings.xml')).iter(ns+'si')]
  dates=[];values=[];observed=[];previous=None;row_count=0;future_cells_skipped=0
  with book.open('xl/worksheets/sheet1.xml') as sheet:
   for _,row in ET.iterparse(sheet,events=('end',)):
    if row.tag!=ns+'row':continue
    row_count+=1
    cells=row.findall(ns+'c')
    if row_count==1:
     assert strs[int(cells[0].find(ns+'v').text)]=='Date'
     names=[strs[int(c.find(ns+'v').text)] for c in cells[1:]]
     dates=[datetime.strptime(h,'ADS_INDEX_%m%d%y').date() for h in names]
     assert len(dates)==1576 and len(set(dates))==1576 and dates==sorted(dates)
     values=[None]*len(dates);observed=[None]*len(dates)
     colmap={c.attrib['r'].rstrip('0123456789'):i for i,c in enumerate(cells[1:])}
     row.clear();continue
    dc=cells[0];dv=dc.find(ns+'v')
    day=datetime.strptime(strs[int(dv.text)],'%Y:%m:%d').date() if dc.attrib.get('t')=='s' else None
    assert day and (previous is None or day>previous);previous=day
    for c in cells[1:]:
     i=colmap.get(c.attrib['r'].rstrip('0123456789'))
     if i is None:raise ValueError('unexpected column')
     if day>dates[i]:future_cells_skipped+=1;continue
     v=c.find(ns+'v')
     if v is None or v.text is None:continue
     try:
      raw=strs[int(v.text)] if c.attrib.get('t')=='s' else v.text
      number=float(raw)
     except (ValueError,IndexError):continue
     if math.isfinite(number):values[i]=number;observed[i]=day
    row.clear()
  rows=[{'vintage_date':d.isoformat(),'observation_date':o.isoformat() if o else None,'ads_value':v} for d,o,v in zip(dates,observed,values)]
  cache.write_text(json.dumps({'source_sha256':plan['source']['raw_sha256'],'rows':rows,'source_row_count':row_count,'future_cells_skipped':future_cells_skipped})+'\n')
finally:tmp.unlink(missing_ok=True)
print('vintages',len(rows),'finite',sum(v is not None for v in values),'source_rows',row_count,'future_cells_skipped',future_cells_skipped)


vintages 1576 finite 1576 source_rows 24338 future_cells_skipped 0


In [2]:
raw=json.loads(cache.read_text());assert raw['source_sha256']==plan['source']['raw_sha256']
versions=pd.DataFrame(raw['rows']);versions['vintage_date']=pd.to_datetime(versions.vintage_date);versions['observation_date']=pd.to_datetime(versions.observation_date)
assert versions.vintage_date.is_unique and versions.vintage_date.is_monotonic_increasing
assert (versions.observation_date<=versions.vintage_date).all()
qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1,expression_cache=None,dataset_cache=None)
calendar=pd.DatetimeIndex(D.calendar(start_time='2011-01-03',end_time='2026-10-07',freq='day'))
train_start,train_end=map(pd.Timestamp,plan['cohorts']['train']);held_start,held_end=map(pd.Timestamp,plan['cohorts']['held_reused'])
train=calendar[(calendar>=train_start)&(calendar<=train_end)];held=calendar[(calendar>=held_start)&(calendar<=held_end)]
assert len(train)==plan['cohorts']['expected_train_sessions'] and len(held)==plan['cohorts']['expected_held_sessions']
signals=train.append(held)
source_dates=versions.vintage_date.to_numpy(dtype='datetime64[ns]');positions=np.searchsorted(source_dates,signals.to_numpy(dtype='datetime64[ns]'),side='left')-1
assert (positions>=0).all()
mapped=versions.iloc[positions].reset_index(drop=True);mapped.insert(0,'signal_date',signals)
mapped['vintage_age_days']=(mapped.signal_date-mapped.vintage_date).dt.days
mapped['observation_age_days']=(mapped.signal_date-mapped.observation_date).dt.days
assert mapped.vintage_age_days.ge(1).all() and mapped.observation_age_days.ge(mapped.vintage_age_days).all()
assert mapped.ads_value.notna().all(), 'unexpected diagonal missingness; stop before DatasetH'
frame=mapped.set_index('signal_date')[['ads_value','vintage_age_days','observation_age_days']].rename(columns={'ads_value':plan['feature']})
frame.index=pd.MultiIndex.from_arrays([signals,['SOXL_SOXX_CASH']*len(signals)],names=['datetime','instrument'])
static=pd.concat({'feature':frame[[plan['feature']]],'meta':frame[['vintage_age_days','observation_age_days']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':tuple(plan['cohorts']['train']),'held':tuple(plan['cohorts']['held_reused'])})
tr=dataset.prepare('train',col_set=['feature','meta']);he=dataset.prepare('held',col_set=['feature','meta'])
assert len(tr)==len(train) and len(he)==len(held)
audit={'status':'source_calendar_availability_complete_no_fit','source_sha256':plan['source']['raw_sha256'],'vintage_count':len(versions),'finite_diagonal_vintages':int(versions.ads_value.notna().sum()),'train_signal_sessions':len(tr),'held_signal_sessions':len(he),'train_nonmissing':int(mapped.iloc[:len(train)].ads_value.notna().sum()),'held_nonmissing':int(mapped.iloc[len(train):].ads_value.notna().sum()),'max_vintage_age_days':int(mapped.vintage_age_days.max()),'max_observation_age_days':int(mapped.observation_age_days.max()),'first_signal_vintage':str(mapped.iloc[0].vintage_date.date()),'last_signal_vintage':str(mapped.iloc[-1].vintage_date.date()),'future_cells_skipped':raw['future_cells_skipped'],'historical_exact_intraday_release_clock_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'availability_audit.json').write_text(json.dumps(audit,indent=2)+'\n')
print(audit)


{'status': 'source_calendar_availability_complete_no_fit', 'source_sha256': '6b6001da2127ad94066828b3a3151368adeb8f83a7dfe65f1901678703cfef44', 'vintage_count': 1576, 'finite_diagonal_vintages': 1576, 'train_signal_sessions': 3020, 'held_signal_sessions': 944, 'train_nonmissing': 3020, 'held_nonmissing': 944, 'max_vintage_age_days': 20, 'max_observation_age_days': 26, 'first_signal_vintage': '2010-12-30', 'last_signal_vintage': '2026-10-02', 'future_cells_skipped': 0, 'historical_exact_intraday_release_clock_qualified': False, 'market_outcome_reads': 0, 'model_fits': 0, 'native_accounts': 0}


In [3]:
import pickle,tarfile,subprocess,shlex
from src.data.nas_market_data import archive_nas_recorder,_settings
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
dataset.config(dump_all=True,recursive=True)
roundtrip=pickle.loads(pickle.dumps(dataset))
assert len(roundtrip.prepare('train',col_set=['feature','meta']))==len(train)
tracking=card/'qlib_recorder'
with R.start(experiment_name='stage1_ads_availability_20261023',uri=str(tracking)):
 rec=R.get_recorder()
 rec.save_objects(artifact_path='inputs',plan=plan,source_sha256=plan['source']['raw_sha256'])
 rec.save_objects(artifact_path='analysis',diagonal_vintages=versions,mapped_signals=mapped,availability_audit=audit)
 rec.save_objects(artifact_path='metadata',dataset=dataset)
 rec.log_metrics(vintage_count=len(versions),train_sessions=len(tr),held_sessions=len(he),model_fits=0,native_accounts=0)
 recorder_id=rec.id;experiment_id=str(rec.experiment_id)
with R.uri_context(str(tracking)):
 checked=R.get_recorder(recorder_id=recorder_id,experiment_id=experiment_id)
 assert len(checked.load_object('metadata/dataset').prepare('held',col_set=['feature','meta']))==len(held)
 assert checked.load_object('analysis/mapped_signals').vintage_date.equals(mapped.vintage_date)
settings=_settings(root);binding=card/'nas_binding.json'
binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':'stage1_ads_availability_20261023','ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=recorder_id,experiment_id=experiment_id)
with tempfile.TemporaryDirectory(prefix='ads-map-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:
  run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as stored:
  names=[m.name for m in stored.getmembers() if m.isfile()]
 assert any(n.endswith('/artifacts/metadata/dataset') for n in names)
summary={'experiment':'stage1_ads_availability_20261023','decision':'source_calendar_alignment_complete_information_untested','objective_achieved':False,'plan':'plan.json','result':'availability_audit.json','source_raw_sha256':plan['source']['raw_sha256'],'train_signal_sessions':len(tr),'held_signal_sessions':len(he),'model_fits':0,'native_accounts':0,'recorder_id':recorder_id,'experiment_id':experiment_id,'recorder_uri':str(tracking),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'artifact_policy':plan['artifact_policy']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print({'recorder_id':recorder_id,'nas_sha256':evidence['sha256']})


{'recorder_id': '5707514424bb4d338c85ecb5a227cb7b', 'nas_sha256': '2a9cb5253904e3b80e1443af3dce70bd968b02b84aad4287caaac238b8dec386'}


## 中文结论

按原始版本严格次一交易日映射，1,576 个历史版本均有有效对角值；Stage1 Qlib 训练 3,020/3,020、复用诊断 944/944 个交易日非缺失。版本日均早于决策日，最大版本日龄 20 天、观测日龄 26 天。DatasetH/Recorder 读回及 NAS 归档恢复通过，Recorder `5707514424bb4d338c85ecb5a227cb7b`。历史盘中首发时刻未逐版确认，采用保守次交易日规则。零收益读取、零拟合、零账户；ADS 信息增量与 SOXX 经济超额尚未检验。下一步冻结匹配原 Stage1 39 输入的信息检验。
